# memopro CUDA check (pre-release gate V2, docs/research/0044)

Checks on a real NVIDIA GPU what the Mac could not: `doctor` with CUDA, census coverage on CUDA,
and hibernation (`host`, `source`, backward in flight) with bit-exact restores.

1. **Runtime -> Change runtime type -> T4 GPU**.
2. Upload the Linux wheel (`memopro-0.1.0a1-cp311-abi3-manylinux_*.whl`, from the *Release*
   workflow's manual run) with the file panel on the left, into `/content`.
3. **Runtime -> Run all**. Copy the JSON printed by the last cell back to the conversation.

Every section records its own result; a failing section does not stop the others.
Run 2 (0045) also checks: `host` reports added host RAM as a negative `rss`, and census
counts the cuBLAS workspace (compare `framework_workspace` with `independent_workspace`).

Run 3 (0049) checks the guards of the defect sweep (0048) and the torch.compile fix of 0049 on
CUDA: in `guards_cuda`, `guards_gpt2` and `guards_compile` every check must say `"ok": true`.
`guards_compile` compiles with inductor, so it takes a few minutes.
Use the wheel sent with run 3 (0049): the first output must show `"guards_0049": true`.

In [ ]:
import glob, json, subprocess, sys, traceback

wheels = sorted(glob.glob("/content/memopro-*.whl"))
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", wheels[-1]], check=True)
import memopro, torch  # noqa: E402

import importlib, importlib.util, os, transformers  # noqa: E401, E402

results = {"memopro": memopro.__version__, "python": sys.version.split()[0], "torch": torch.__version__,
           "transformers": transformers.__version__,
           "wheel": os.path.basename(wheels[-1]) if wheels else None,
           "guards_0049": importlib.util.find_spec("memopro.hibernate._guards") is not None
           and hasattr(importlib.import_module("memopro.hibernate._guards"), "_compiled_wrappers"),
           "cuda": torch.cuda.is_available(),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}


def section(name):
    def wrap(fn):
        try:
            results[name] = fn()
        except Exception as e:  # keep going: report everything
            results[name] = {"error": f"{type(e).__name__}: {e}", "trace": traceback.format_exc()[-800:]}
        print(name, "->", json.dumps(results[name], default=str)[:300])
    return wrap
print(results)

In [ ]:
@section("doctor")
def _():
    r = memopro.doctor()
    print(r.summary())
    cuda = [d for d in r.env.devices if d.kind == "cuda"]
    return {"cuda_devices": len(cuda), "device_budget": r.budget.device,
            "free_reported": cuda[0].available_bytes if cuda else None,
            "free_torch": torch.cuda.mem_get_info(0)[0] if torch.cuda.is_available() else None,
            "notes": r.warnings()}

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

@section("census_cuda")
def _():
    if not torch.cuda.is_available():
        return "skipped: no CUDA"
    torch.manual_seed(0)
    m = GPT2LMHeadModel(GPT2Config(n_layer=4, n_embd=256, n_head=4, vocab_size=8000)).cuda()
    opt = torch.optim.AdamW(m.parameters())
    x = torch.randint(0, 8000, (4, 64), device="cuda")
    for _ in range(2):
        opt.zero_grad(); m(x, labels=x).loss.backward(); opt.step()
    # independent check of the 0045 F2 hypothesis: cuBLAS workspaces live in the allocator
    torch.cuda.synchronize()
    a0 = torch.cuda.memory_allocated()
    torch._C._cuda_clearCublasWorkspaces()
    independent_workspace = a0 - torch.cuda.memory_allocated()
    opt.zero_grad(); m(x, labels=x).loss.backward(); opt.step()  # workspace comes back
    with memopro.census.record(m, opt, mode="light") as c:
        opt.zero_grad(); m(x, labels=x).loss.backward(); opt.step()
    print(c.summary())
    j = c.to_json()
    return {"coverage_cuda": j["coverage_at_end"].get("cuda"),
            "unattributed_cuda": j["unclassified_at_end"].get("cuda"),
            "framework_workspace": j.get("framework_workspace"),
            "independent_workspace": independent_workspace,
            "bytes": {k: v["bytes"] for k, v in j["categories"].items()}}


In [ ]:
def cuda_mem():
    torch.cuda.synchronize(); torch.cuda.empty_cache()
    return torch.cuda.memory_reserved()

@section("hibernate_gpt2")
def _():
    if not torch.cuda.is_available():
        return "skipped: no CUDA"
    out = {}
    model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").cuda().eval()
    x = torch.randint(0, 50257, (1, 64), device="cuda")
    with torch.no_grad():
        ref = model(x).logits.float().cpu()
    for mode in ("host", "auto"):
        before = cuda_mem()
        h = memopro.hibernate.now(model, mode=mode)
        freed = before - cuda_mem()
        with torch.no_grad():
            got = model(x).logits.float().cpu()  # forward wakes it
        out[mode] = {"modes": h.bytes_by_mode(), "kept": {k: len(v) for k, v in h.kept.items()},
                     "cuda_reserved_freed": freed, "reclaimed_report": h.reclaimed,
                     "bit_exact": bool(torch.equal(got, ref)), "asleep_after_forward": h.asleep}
    return out

In [ ]:
@section("backward_in_flight")
def _():
    if not torch.cuda.is_available():
        return "skipped: no CUDA"
    torch.manual_seed(0)
    make = lambda: torch.nn.Sequential(torch.nn.Linear(512, 512), torch.nn.Tanh(), torch.nn.Linear(512, 1)).cuda()
    a, b = make(), make(); b.load_state_dict(a.state_dict())
    x = torch.randn(8, 512, device="cuda", requires_grad=True)
    b(x).sum().backward()
    loss = a(x).sum()
    h = memopro.hibernate.now(a, mode="host")
    loss.backward()
    return {"grads_equal": all(torch.equal(p.grad, q.grad) for p, q in zip(a.parameters(), b.parameters())),
            "kept": {k: len(v) for k, v in h.kept.items()}}

In [ ]:
# Run 3 (0049): guards on CUDA. Anything that reads, saves, copies, moves or steps a sleeping
# object must wake it first; interruptions, threads and deletion must not lose data (0048).
import copy, gc, io, pickle, threading, weakref, zipfile  # noqa: E401
from memopro import hibernate
from memopro.hibernate import _guards

DEV, MODE = ("cuda", "host") if torch.cuda.is_available() else (None, None)


def small(seed=0):
    torch.manual_seed(seed)
    layers = [torch.nn.Linear(256, 256), torch.nn.Tanh(), torch.nn.Linear(256, 16)]
    return torch.nn.Sequential(*layers).to(DEV)


def same(a, b):
    a, b = list(a), list(b)
    return len(a) == len(b) and all(torch.equal(x, y) for x, y in zip(a, b))


def dev_alloc():
    if DEV != "cuda":
        return 0
    torch.cuda.synchronize()
    return torch.cuda.memory_allocated()


def sleep(obj, mode=None):
    """Hibernate and prove that something really went to sleep, so no check passes vacuously."""
    h = hibernate.now(obj, mode=mode or MODE)
    assert h.asleep and h.nbytes > 0, repr(h)
    return h


def run_checks(checks):
    out = {}
    for name, fn in checks.items():
        try:
            ok, detail = fn()
            out[name] = {"ok": bool(ok), **detail}
        except Exception as e:
            out[name] = {"ok": False, "error": f"{type(e).__name__}: {e}"[:300]}
        finally:  # the next check starts from a clean state
            for h in hibernate.handles():
                if h.asleep:
                    try:
                        h.wake()
                    except Exception:
                        h.discard()
        print(f"  {name}: {'OK' if out[name]['ok'] else 'FAIL'}", json.dumps(out[name], default=str)[:200])
    all_ok = all(v["ok"] for v in out.values())
    out["all_ok"] = all_ok
    return out

In [ ]:
def c_state_dict():
    m = small()
    ref = {k: v.clone() for k, v in m.state_dict().items()}
    h = sleep(m)
    sd = m.state_dict()
    on_dev = all(v.device.type == DEV for v in sd.values())
    return same(sd.values(), ref.values()) and on_dev, {"slept": h.bytes_by_mode()}


def c_torch_save():
    m, x = small(), torch.randn(4, 256, device=DEV)
    with torch.no_grad():
        ref = m(x)
    sleep(m)
    buf = io.BytesIO()
    torch.save(m, buf)
    pkl = [zipfile.ZipFile(buf).read(n) for n in zipfile.ZipFile(buf).namelist() if n.endswith(".pkl")]
    clean = bool(pkl) and all(b"memopro" not in p for p in pkl)  # no hook or handle pickled
    buf.seek(0)
    with torch.no_grad():
        got = torch.load(buf, weights_only=False)(x)
    return torch.equal(got, ref) and clean, {"no_memopro_in_pickle": clean}


def c_deepcopy():
    m, x = small(), torch.randn(4, 256, device=DEV)
    with torch.no_grad():
        ref = m(x)
    sleep(m)
    c = copy.deepcopy(m)
    with torch.no_grad():
        got = c(x)
    marks = sum("_memopro_handle" in p.__dict__ for p in c.parameters())
    return torch.equal(got, ref) and marks == 0, {"copied_marks": marks}


def c_pickle_tensor():
    t = torch.randn(100_000, device=DEV)
    ref = t.clone()
    sleep(t)
    t2 = pickle.loads(pickle.dumps(t))
    ok = torch.equal(t2, ref) and t2.device.type == DEV and "__reduce_ex__" not in t.__dict__
    return ok, {"elements": t2.numel()}


def c_load_state_dict():
    m, src = small(0), small(1)
    sleep(m)
    m.load_state_dict(src.state_dict())
    return same(m.state_dict().values(), src.state_dict().values()), {}


def c_move():
    m = small()
    ref = [p.detach().cpu().clone() for p in m.parameters()]
    sleep(m)
    m.cpu()
    to_cpu = all(p.device.type == "cpu" for p in m.parameters()) and same(m.parameters(), ref)
    m = small()
    ref = [p.detach().half() for p in m.parameters()]
    sleep(m)
    m.half()
    to_half = all(p.dtype == torch.float16 for p in m.parameters()) and same(m.parameters(), ref)
    return to_cpu and to_half, {"to_cpu": to_cpu, "to_half": to_half}


def c_count():
    m = small()
    n = sum(p.numel() for p in m.parameters())
    sleep(m)
    return sum(p.numel() for p in m.parameters()) == n, {"params": n}


def c_step(kind):
    def check():  # sleep between backward and step; compare with the same step on an awake twin
        a, b = small(), small()
        oa = torch.optim.AdamW(a.parameters(), lr=1e-2, **{kind: True})
        ob = torch.optim.AdamW(b.parameters(), lr=1e-2, **{kind: True})
        a(torch.randn(8, 256, device=DEV)).square().mean().backward()
        for p, q in zip(a.parameters(), b.parameters()):
            q.grad = p.grad.clone()
        h = sleep(a)  # parameters and gradients
        oa.step()
        ob.step()
        return same(a.parameters(), b.parameters()), {"slept": h.bytes_by_mode()}
    return check


def train(sleep_between, steps=6):
    m = small()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    g = torch.Generator(device=DEV).manual_seed(1)
    freed = []
    for _ in range(steps):
        m(torch.randn(16, 256, device=DEV, generator=g)).square().mean().backward()
        opt.step()
        opt.zero_grad()
        if sleep_between:  # model and optimizer state sleep between steps
            before = dev_alloc()
            sleep(m)
            sleep(opt)
            freed.append(before - dev_alloc())
    return [p.detach().clone() for p in m.parameters()], freed


def c_training():
    plain, _ = train(False)
    control, _ = train(False)
    slept, freed = train(True)
    return same(slept, plain), {"control_deterministic": same(plain, control), "device_freed": freed[:2]}


def c_interrupt():  # Ctrl-C on the 3rd tensor: the ones already emptied must come back
    m = small()
    ref = [p.detach().clone() for p in m.parameters()]
    orig, calls = hibernate._sleep, [0]

    def flaky(*args, **kwargs):
        calls[0] += 1
        if calls[0] == 3:
            raise KeyboardInterrupt
        return orig(*args, **kwargs)

    hibernate._sleep = flaky
    try:
        hibernate.now(m, mode=MODE)
        interrupted = False
    except KeyboardInterrupt:
        interrupted = True
    finally:
        hibernate._sleep = orig
    intact = same(m.parameters(), ref) and all(p.device.type == DEV for p in m.parameters())
    clean = not any("_memopro_handle" in p.__dict__ for p in m.parameters())
    clean = clean and all(h.object is not m for h in hibernate.handles())
    return interrupted and intact and clean, {"interrupted": interrupted, "intact": intact, "clean": clean}


def c_threads():
    x, errs = torch.randn(4, 256, device=DEV), []

    def run(n, fn):
        ts = [threading.Thread(target=fn) for _ in range(n)]
        for t in ts:
            t.start()
        for t in ts:
            t.join()

    m = small()
    with torch.no_grad():
        ref = m(x)
    sleep(m)
    outs = []

    def call():  # 8 threads call the sleeping model at once
        try:
            with torch.no_grad():
                outs.append(m(x))
        except Exception as e:
            errs.append(repr(e)[:120])

    run(8, call)
    calls_ok = len(outs) == 8 and all(torch.equal(o, ref) for o in outs)
    m2 = small()
    with torch.no_grad():
        ref2 = m2(x)
    hs = []

    def hib():  # 6 threads hibernate the same model at once (0048 T1)
        try:
            hs.append(hibernate.now(m2, mode=MODE))
        except Exception as e:
            errs.append(repr(e)[:120])

    run(6, hib)
    with torch.no_grad():
        got2 = m2(x)
    now_ok = len(hs) == 6 and len({id(h) for h in hs}) == 1 and torch.equal(got2, ref2)
    return calls_ok and now_ok and not errs, {"calls_ok": calls_ok, "concurrent_now_ok": now_ok, "errors": errs[:2]}


def c_lifetime():
    n0 = len(hibernate.handles())
    m = small()
    mr, hr = weakref.ref(m), weakref.ref(sleep(m))
    del m
    gc.collect()
    asleep_freed = mr() is None and hr() is None  # its host copy goes with the handle
    m = small()
    mr = weakref.ref(m)
    sleep(m).wake()
    del m
    gc.collect()
    awake_freed = mr() is None
    ok = asleep_freed and awake_freed and len(hibernate.handles()) == n0
    return ok, {"asleep_freed": asleep_freed, "awake_freed": awake_freed}


def c_clean():
    gc.collect()
    return not _guards._owners, {"owners": len(_guards._owners), "global_hook": _guards._global_hook is not None}


@section("guards_cuda")
def _():
    if DEV is None:
        return "skipped: no CUDA"
    return run_checks({
        "state_dict": c_state_dict, "torch_save": c_torch_save, "deepcopy": c_deepcopy,
        "pickle_tensor": c_pickle_tensor, "load_state_dict": c_load_state_dict, "move": c_move,
        "count": c_count, "step_foreach": c_step("foreach"), "step_fused": c_step("fused"),
        "training": c_training, "interrupt": c_interrupt, "threads": c_threads,
        "lifetime": c_lifetime, "clean_state": c_clean,
    })

In [ ]:
import tempfile


@section("guards_gpt2")
def _():  # a real checkpoint: mostly `source` (re-read from the safetensors file) on CUDA
    if DEV is None:
        return "skipped: no CUDA"
    model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").to(DEV).eval()
    x = torch.randint(0, 50257, (1, 32), device=DEV, generator=torch.Generator(device=DEV).manual_seed(0))
    gen_args = {"max_new_tokens": 16, "do_sample": False, "pad_token_id": 50256}
    with torch.no_grad():
        ref = model(x).logits
        ref_gen = model.generate(x, **gen_args)
    ref_sd = {k: v.detach().clone() for k, v in model.state_dict().items()}
    first = {}

    def auto():
        before = dev_alloc()
        h = sleep(model, mode="auto")
        if not first:
            first.update(modes=h.bytes_by_mode(), kept={k: len(v) for k, v in h.kept.items()},
                         device_freed=before - dev_alloc())
        return h

    def c_generate():  # generate() reads model.device first: parameters() must wake it
        auto()
        with torch.no_grad():
            gen = model.generate(x, **gen_args)
        return torch.equal(gen, ref_gen), {"tokens": gen.shape[-1]}

    def c_state_dict():
        auto()
        return same(model.state_dict().values(), ref_sd.values()), {}

    def c_save_pretrained():
        auto()
        with tempfile.TemporaryDirectory() as d:
            model.save_pretrained(d)
            again = GPT2LMHeadModel.from_pretrained(d).to(DEV).eval()
        with torch.no_grad():
            got = again(x).logits
        return torch.equal(got, ref), {}

    def c_step():  # last: it changes the weights. Params sleep as `source`, grads as `host`
        twin = copy.deepcopy(model)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
        opt_twin = torch.optim.AdamW(twin.parameters(), lr=1e-4)
        model(x, labels=x).loss.backward()
        for p, q in zip(model.parameters(), twin.parameters()):
            q.grad = p.grad.clone()
        h = sleep(model, mode="auto")
        opt.step()
        opt_twin.step()
        return same(model.parameters(), twin.parameters()), {"slept": h.bytes_by_mode()}

    out = run_checks({"generate": c_generate, "state_dict": c_state_dict,
                      "save_pretrained": c_save_pretrained, "step": c_step})
    out["first_sleep"] = first
    return out

In [ ]:
from torch._dynamo.utils import counters
import warnings


def fresh():
    torch._dynamo.reset()
    counters.clear()


def breaks():
    return sum(counters["graph_break"].values())


def memopro_warnings(record):
    return sum("memopro" in str(w.message) for w in record)


def k_compiled(kind):
    def check():  # hibernate the original model; the compiled call must keep its graph
        fresh()
        m, x = small(), torch.randn(64, 256, device=DEV)
        if kind == "module_compile":
            m.compile()
            call = m
        elif kind == "reduce_overhead":  # CUDA graphs: replays record tensor addresses
            call = torch.compile(m, mode="reduce-overhead")
        else:
            call = torch.compile(m, fullgraph=kind == "fullgraph")
        with torch.no_grad():
            for _ in range(3):  # warm up (CUDA graphs record on a later call)
                ref = call(x).clone()  # clone: CUDA graph outputs are overwritten by the next run
        n = counters["stats"]["unique_graphs"]
        ok = True
        with warnings.catch_warnings(record=True) as record:
            warnings.simplefilter("always")
            for _ in range(2):
                sleep(m)
                with torch.no_grad():  # asleep, then awake: both give the compiled output
                    ok = ok and torch.equal(call(x).clone(), ref) and torch.equal(call(x).clone(), ref)
        detail = {"graph_breaks": breaks(), "new_graphs": counters["stats"]["unique_graphs"] - n,
                  "warnings": memopro_warnings(record)}
        return ok and detail["graph_breaks"] == 0 and detail["new_graphs"] == 0 and detail["warnings"] == 0, detail
    return check


def k_unseen():  # compiled after hibernation: memopro cannot see it -> correct, and one warning
    fresh()
    m, x = small(), torch.randn(64, 256, device=DEV)
    with torch.no_grad():
        ref = m(x)
    sleep(m)
    cm = torch.compile(m)
    with warnings.catch_warnings(record=True) as record:
        warnings.simplefilter("always")
        with torch.no_grad():
            got = cm(x)
    return torch.equal(got, ref) and memopro_warnings(record) == 1, {"warnings": memopro_warnings(record)}


def compiled_training(sleep_between):
    m = small()
    cm = torch.compile(m)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    g = torch.Generator(device=DEV).manual_seed(1)
    for _ in range(4):
        cm(torch.randn(16, 256, device=DEV, generator=g)).square().mean().backward()
        opt.step()
        opt.zero_grad()
        if sleep_between:
            sleep(m)
            sleep(opt)
    return [p.detach().clone() for p in m.parameters()]


def k_training():
    fresh()
    plain = compiled_training(False)
    control = compiled_training(False)
    counters.clear()
    slept = compiled_training(True)
    return same(slept, plain), {"control_deterministic": same(plain, control), "graph_breaks": breaks()}


def k_optimizer_step():  # torch.compile(opt.step) while parameters and gradients sleep
    fresh()
    a, b = small(), small()
    oa = torch.optim.AdamW(a.parameters(), lr=1e-2)
    ob = torch.optim.AdamW(b.parameters(), lr=1e-2)
    step_a, step_b = torch.compile(oa.step), torch.compile(ob.step)
    a(torch.randn(8, 256, device=DEV)).square().mean().backward()
    for p, q in zip(a.parameters(), b.parameters()):
        q.grad = p.grad.clone()
    sleep(a)
    step_a()
    step_b()
    return same(a.parameters(), b.parameters()), {}


@section("guards_compile")
def _():  # torch.compile (inductor) with hibernation, found and fixed in 0049
    if DEV is None:
        return "skipped: no CUDA"
    out = run_checks({
        "compiled_wrapper": k_compiled("wrapper"), "fullgraph": k_compiled("fullgraph"),
        "module_compile": k_compiled("module_compile"), "reduce_overhead": k_compiled("reduce_overhead"),
        "compiled_after_hibernate": k_unseen,
        "training": k_training, "optimizer_step": k_optimizer_step,
    })
    torch._dynamo.reset()
    return out

In [ ]:
print(json.dumps(results, indent=2, default=str))